In [3]:
import pandas as pd
import faiss
import numpy as np
from docx import Document
from sentence_transformers import SentenceTransformer
from langchain_community.chat_models import ChatOllama

e:\PMS\CBE_PMS_PROJECT\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
llm = ChatOllama(model="llama3", temperature=0.3)
def generate_response(prompt):
    response = llm.invoke(prompt)
    return response.content

C:\Users\user\AppData\Local\Temp\ipykernel_16144\3528689757.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3", temperature=0.3)


In [5]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2637.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [123]:
bsc_path = "../Data/raw/Bsc.xlsx"  # update path

bsc_df = pd.read_excel(bsc_path)

bsc_df.head()

,Major Objectives,Major–KPI,Target
0,Ensure Sustainable Profitability,Economic Profit,Achieve Economic Profit Plan
1,Ensure Sustainable Profitability,Economic Profit,Graduating branches from loss to profit making
2,Enhance Resource Mobilization,Incremental Deposit,Achieve Incremental Deposit plan
3,Enhance Resource Mobilization,Incremental Deposit,Mobilize Deposit from Retail Customers
4,Enhance Resource Mobilization,Incremental Deposit,Mobilize Deposit from Wholesale Customers


In [126]:
def chunk_bsc_enhanced(df):
    """
    Create meaningful BSC chunks with strategic context
    Each chunk combines: Major Objective + KPI + Target
    """
    chunks = []
    
    for idx, row in df.iterrows():
        # Skip if major objective is missing
        if pd.isna(row['Major Objectives']):
            continue
        
        # Create rich chunk text
        chunk_text = f"""
STRATEGIC AREA: {row['Major Objectives']}
KEY PERFORMANCE INDICATOR: {row['Major–KPI']}
TARGET: {row['Target']}
"""
        
        # Create metadata for retrieval
        chunks.append({
            'id': f"bsc_{idx}",
            'source': 'BSC',
            'strategic_area': row['Major Objectives'],
            'kpi': row['Major–KPI'],
            'target': row['Target'],
            'text': chunk_text.strip(),
            'chunk_type': 'strategic_goal',
            'chunk_length': len(chunk_text)
        })
    
    print(f"✓ BSC: {len(chunks)} chunks created")
    return chunks
bsc_chunks = chunk_bsc_enhanced(bsc_df)

✓ BSC: 40 chunks created


In [93]:
import re

def extract_responsibilities(text):
    """
    Extract bullet responsibilities from JD text.
    """

    # Find section
    pattern = r"key job duties and responsibilities:(.*?)(iv\.|v\.|vi\.|$)"
    match = re.search(pattern, text, re.IGNORECASE)

    if not match:
        return []

    section = match.group(1)

    # Split bullet points
    bullets = re.split(r"[•\-]\s*", section)

    # Clean bullets
    responsibilities = [
        b.strip() for b in bullets
        if len(b.strip()) > 20  # remove noise
    ]

    return responsibilities

In [108]:
import pandas as pd
from docx import Document
import re
def extract_all_jd_tables(file_path):
    from docx import Document

    doc = Document(file_path)
    jd_records = []

    for table in doc.tables:
        table_text = ""

        for row in table.rows:
            row_text = " | ".join([cell.text.strip() for cell in row.cells if cell.text.strip()])
            table_text += row_text + "\n"

        table_text = table_text.lower()
        table_text = re.sub(r'\s+', ' ', table_text)

        def extract(field):
            pattern = rf"{field}\s*:\s*([^|]+)"
            match = re.search(pattern, table_text)
            return match.group(1).strip() if match else None

        responsibilities = extract_responsibilities(table_text)

        jd_records.append({
            "job_title": extract("job title"),
            "division": extract("division"),
            "department": extract("department"),
            "job_grade": extract("job grade"),
            "job_category": extract("job category"),
            "job_objective": extract("job objective"),
            "responsibilities": responsibilities,
            "full_text": table_text
        })

    df = pd.DataFrame(jd_records)

    return df

In [109]:
jd_path = "../Data/raw/JD2.docx"
jd_df = extract_all_jd_tables(jd_path)

print(jd_df["responsibilities"][0][:5])

['the job holder’s responsibilities include, but are not limited to: plan, coordinate and manage the overall banking operation activities in the branch; set annual operational plan and operational budget of the branch for retail, wholesale and ifb businesses; and digital banking channels; create awareness and clarity to staff of the branch about the bank’s strategy, plans and corporate views; ensure required resources are satisfied at the branch; conduct daily sales and operational huddle and weekly meetings with staff of the branch; establish and lead sales, service, and operation teams of the branch; oversee lobby area is managed by staff of the bank with good knowledge of products & services and having good communication skill; oversee the branch maintains sufficient cash balance within the cash holding limits; oversee security operations and personnel’s of the branch; make operational risk identification, assessment, measurement and put in place controlling and mitigation mechanism

In [110]:
jd_df.head()

,job_title,division,department,job_grade,job_category,job_objective,responsibilities,full_text
0,branch manager special branch,retail & branch banking,district,15 unit: branch,middle level management: mlm,to ensure the overall banking operation activi...,"[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
1,branch business manager,retail & branch banking,district,13 unit: branch,operative level management: olm,to ensure the effectiveness of the branch sale...,"[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
2,customer service manager -sales,retail & branch banking,district,12 unit: branch,operative level management: olm,"to sale bank products and services, mobilizes ...",[the job holder’s responsibilities include but...,job details /profile/: | job details /profile/...
3,customer service manager -service,retail & branch banking,district,12 unit: branch,operative level management: olm,"to serve the bank’s customers, mobilizes resou...","[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
4,senior branch banking officer-sales,retail & branch banking,district,11 unit: branch,experienced professional: ep,"to sale bank products and services, mobilizes ...","[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...


In [ ]:
def chunk_job_description_simple(jd_row):
    """
    Simpler version - more robust for various data types
    """
    chunks = []
    role_name = str(jd_row.get('job_title', 'Unknown')).strip()
    
    # Skip if role name is invalid
    if role_name == 'Unknown' or role_name == 'nan':
        return chunks
    
    # CHUNK 1: Header
    header_text = f"""
ROLE: {role_name}
DIVISION: {jd_row.get('division', 'N/A')}
DEPARTMENT: {jd_row.get('department', 'N/A')}
JOB GRADE: {jd_row.get('job_grade', 'N/A')}
JOB CATEGORY: {jd_row.get('job_category', 'N/A')}
"""
    chunks.append({
        'id': f"jd_{role_name.replace(' ', '_')}_header",
        'source': 'JD',
        'role': role_name,
        'chunk_type': 'header',
        'text': header_text.strip()
    })
    
    # CHUNK 2: Job Objective
    job_objective = jd_row.get('job_objective')
    if job_objective and str(job_objective) != 'nan':
        objective_text = f"""
ROLE: {role_name}
JOB OBJECTIVE: {job_objective}
"""
        chunks.append({
            'id': f"jd_{role_name.replace(' ', '_')}_objective",
            'source': 'JD',
            'role': role_name,
            'chunk_type': 'objective',
            'text': objective_text.strip()
        })
    
    # CHUNK 3: Full responsibilities as single chunk (safer)
    responsibilities = jd_row.get('responsibilities')
    
    if responsibilities is not None and str(responsibilities) != 'nan':
        # Convert to string safely
        resp_str = str(responsibilities)
        
        # Clean up the string
        resp_str = resp_str.replace('[', '').replace(']', '').replace("'", "")
        
        # Split by common delimiters
        if ';' in resp_str:
            resp_items = resp_str.split(';')
        elif ',' in resp_str:
            resp_items = resp_str.split(',')
        else:
            resp_items = [resp_str]
        
        # Clean each item
        resp_items = [r.strip() for r in resp_items if r.strip() and len(r.strip()) > 10]
        
        if resp_items:
            # Option A: Single chunk with all responsibilities
            resp_text = f"""
ROLE: {role_name}
KEY RESPONSIBILITIES:
""" + "\n".join([f"• {r}" for r in resp_items[:15]])  # Limit to 15 items
            
            chunks.append({
                'id': f"jd_{role_name.replace(' ', '_')}_responsibilities",
                'source': 'JD',
                'role': role_name,
                'chunk_type': 'responsibilities',
                'text': resp_text.strip()
            })
            
            # Option B: Also create a shorter summary for matching (first 5 responsibilities)
            if len(resp_items) > 5:
                summary_text = f"""
ROLE: {role_name}
KEY RESPONSIBILITIES SUMMARY:
""" + "\n".join([f"• {r}" for r in resp_items[:5]])
                
                chunks.append({
                    'id': f"jd_{role_name.replace(' ', '_')}_summary",
                    'source': 'JD',
                    'role': role_name,
                    'chunk_type': 'summary',
                    'text': summary_text.strip()
                })
    
    return chunks
def chunk_all_job_descriptions_simple(df):
    """Simpler version - more robust"""
    all_chunks = []
    for idx, row in df.iterrows():
        job_title = row.get('job_title')
        if job_title and pd.notna(job_title) and str(job_title) != 'nan':
            role_chunks = chunk_job_description_simple(row)
            all_chunks.extend(role_chunks)
            print(f"  ✓ {job_title}: {len(role_chunks)} chunks")
    
    print(f"\n✓ JD: {len(all_chunks)} total chunks created")
    return all_chunks

In [130]:
# Run the fixed chunking
print("Testing fixed chunking...\n")
chunks = chunk_all_job_descriptions_simple(jd_df)
# Display results
print("\n" + "=" * 50)
print("SAMPLE CHUNKS:")
print("=" * 50)
    
for chunk in chunks[:3]:  # Show first 3 chunks
    print(f"\nChunk ID: {chunk['id']}")
    print(f"Type: {chunk['chunk_type']}")
    print(f"Role: {chunk['role']}")
    print(f"Text: {chunk['text'][:200]}...")
    print("-" * 40)

Testing fixed chunking...

  ✓ branch manager special branch: 4 chunks
  ✓ branch business manager: 4 chunks
  ✓ customer service manager -sales: 3 chunks
  ✓ customer service manager -service: 4 chunks
  ✓ senior branch banking officer-sales: 4 chunks
  ✓ senior branch banking officer-service: 4 chunks
  ✓ branch banking officer: 2 chunks
  ✓ branch banking officer-service: 4 chunks
  ✓ retail customer relationship manager: 4 chunks
  ✓ wholesale customer relationship manager: 4 chunks
  ✓ ifb customer relationship manager: 4 chunks
  ✓ digital channel officer: 4 chunks
  ✓ junior officer ii: 2 chunks
  ✓ junior officer i: 2 chunks
  ✓ bank trainee: 2 chunks
  ✓ branch operation manager: 4 chunks
  ✓ senior branch banking officer: 4 chunks
  ✓ branch banking officer: 2 chunks
  ✓ junior officer ii: 4 chunks
  ✓ junior officer i: 4 chunks
  ✓ branch control manager: 4 chunks
  ✓ branch controller: 2 chunks
  ✓ የገንዘብ ቤት ተላላኪ /cash office service attendant/: 3 chunks

✓ JD: 78 total chun

In [132]:
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# Load your embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def create_faiss_index(chunks, index_name):
    """
    Create FAISS index from chunks
    """
    print(f"\n--- Creating FAISS index: {index_name} ---")
    
    if not chunks:
        print(f"  ⚠ No chunks to index")
        return None, None
    
    # Extract texts from chunks
    texts = [chunk['text'] for chunk in chunks]
    
    # Generate embeddings
    print(f"  Generating embeddings for {len(texts)} chunks...")
    embeddings = embedder.encode(texts, show_progress_bar=True)
    
    # Create FAISS index (using L2 distance)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings.astype('float32'))
    
    print(f"  ✓ Index created with {index.ntotal} vectors (dimension: {dimension})")
    
    return index, chunks


def save_index(index, chunks, filepath):
    """Save FAISS index and chunks to disk"""
    faiss.write_index(index, f"{filepath}.faiss")
    
    with open(f"{filepath}_chunks.pkl", 'wb') as f:
        pickle.dump(chunks, f)
    
    print(f"  ✓ Saved to: {filepath}.faiss and {filepath}_chunks.pkl")


def load_index(filepath):
    """Load FAISS index and chunks from disk"""
    index = faiss.read_index(f"{filepath}.faiss")
    
    with open(f"{filepath}_chunks.pkl", 'rb') as f:
        chunks = pickle.load(f)
    
    return index, chunks


# ============================================
# STEP 4: SEMANTIC SEARCH (Role ↔ BSC Matching)
# ============================================

def semantic_search(index, chunks, query, top_k=5):
    """
    Perform semantic search on an index
    Returns chunks with similarity scores
    """
    # Encode query
    query_embedding = embedder.encode([query])
    
    # Search
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    
    # Convert L2 distance to similarity score (0-1, higher = more similar)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(chunks):
            similarity = 1 / (1 + distances[0][i])
            results.append({
                'chunk': chunks[idx],
                'similarity': similarity,
                'distance': distances[0][i]
            })
    
    return results


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3971.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [133]:
print("=" * 60)
print("STARTING FAISS INDEXING AND SEMANTIC SEARCH")
print("=" * 60)

STARTING FAISS INDEXING AND SEMANTIC SEARCH


In [134]:
# ============================================
# STEP 1: VERIFY YOUR CHUNKS EXIST
# ============================================

print("\n[1/5] Verifying chunks...")

# Check if you have bsc_chunks and jd_chunks from your preprocessing
# Replace these variable names with whatever you used in your preprocessing

try:
    # Option A: If you saved chunks to variables during preprocessing
    # Use the actual variable names from your chunking code
    print(f"  ✓ BSC chunks available: {len(bsc_chunks)} chunks")
    print(f"  ✓ JD chunks available: {len(chunks)} chunks")
except NameError:
    print("  ⚠ Chunks not found in memory")
    print("  Loading from saved files if available...")
    
    # Option B: Load from saved pickle files
    try:
        with open('./chunks/bsc_chunks.pkl', 'rb') as f:
            bsc_chunks = pickle.load(f)
        with open('./chunks/jd_chunks.pkl', 'rb') as f:
            jd_chunks = pickle.load(f)
        print(f"  ✓ Loaded BSC chunks: {len(bsc_chunks)} chunks")
        print(f"  ✓ Loaded JD chunks: {len(jd_chunks)} chunks")
    except:
        print("  ✗ No chunks found. Please run preprocessing first.")
        print("  Make sure you have bsc_chunks and jd_chunks variables")
        exit()


[1/5] Verifying chunks...
  ✓ BSC chunks available: 40 chunks
  ✓ JD chunks available: 78 chunks


In [136]:
# ============================================
# STEP 2: CREATE FAISS INDEXES
# ============================================

print("\n[2/5] Creating FAISS indexes...")

# Create BSC index
bsc_index, bsc_chunks_final = create_faiss_index(bsc_chunks, "BSC")

# Create JD index
jd_index, jd_chunks_final = create_faiss_index(chunks, "JD")

# ============================================
# STEP 3: SAVE INDEXES
# ============================================

print("\n[3/5] Saving indexes to disk...")

import os
os.makedirs("./indexes", exist_ok=True)

# Save BSC
save_index(bsc_index, bsc_chunks_final, "./indexes/bsc_index")

# Save JD
save_index(jd_index, jd_chunks_final, "./indexes/jd_index")

print("\n  ✓ Indexes saved successfully!")



[2/5] Creating FAISS indexes...

--- Creating FAISS index: BSC ---
  Generating embeddings for 40 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:00<00:00,  6.45it/s]


  ✓ Index created with 40 vectors (dimension: 384)

--- Creating FAISS index: JD ---
  Generating embeddings for 78 chunks...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.06s/it]

  ✓ Index created with 78 vectors (dimension: 384)

[3/5] Saving indexes to disk...
  ✓ Saved to: ./indexes/bsc_index.faiss and ./indexes/bsc_index_chunks.pkl
  ✓ Saved to: ./indexes/jd_index.faiss and ./indexes/jd_index_chunks.pkl

  ✓ Indexes saved successfully!


In [143]:

# ============================================
# STEP 4: TEST SEMANTIC SEARCH ON BSC
# ============================================

print("\n[4/5] Testing semantic search on BSC...")

test_queries = [
    "Job Title:  Branch Business Manager",
]

for query in test_queries:
    print(f"\n--- Query: '{query}' ---")
    results = semantic_search(bsc_index, bsc_chunks_final, query, top_k=4)
    
    print(f"  ✓ Found {len(results)} results:")
    print(results)


[4/5] Testing semantic search on BSC...

--- Query: 'Job Title:  Branch Business Manager' ---
  ✓ Found 4 results:
[{'chunk': {'id': 'bsc_1', 'source': 'BSC', 'strategic_area': 'Ensure Sustainable  Profitability', 'kpi': 'Economic Profit', 'target': 'Graduating branches from loss to profit making', 'text': 'STRATEGIC AREA: Ensure Sustainable  Profitability\nKEY PERFORMANCE INDICATOR: Economic Profit\nTARGET: Graduating branches from loss to profit making', 'chunk_type': 'strategic_goal', 'chunk_length': 149}, 'similarity': np.float32(0.46611938), 'distance': np.float32(1.1453731)}, {'chunk': {'id': 'bsc_16', 'source': 'BSC', 'strategic_area': 'Ensure Prudent Resource Allocation', 'kpi': 'Loan Disbursement', 'target': 'Achieve Loan Disbursement Plan', 'text': 'STRATEGIC AREA: Ensure Prudent Resource Allocation\nKEY PERFORMANCE INDICATOR: Loan Disbursement\nTARGET: Achieve Loan Disbursement Plan', 'chunk_type': 'strategic_goal', 'chunk_length': 136}, 'similarity': np.float32(0.42385527)

In [ ]:
def match_role_to_bsc(role_name, jd_chunks, bsc_index, bsc_chunks, top_k=5):
    """
    Match a specific role to relevant BSC objectives
    """
    print(f"\n{'='*50}")
    print(f"MATCHING ROLE: {role_name}")
    print(f"{'='*50}")
    
    # Find the JD chunk for this role (summary or header chunk is best for matching)
    role_query = None
    for chunk in jd_chunks:
        if chunk['role'] == role_name and chunk['chunk_type'] in ['summary', 'header', 'objective']:
            role_query = chunk['text']
            break
    
    if not role_query:
        # If no specific chunk found, use first chunk of this role
        for chunk in jd_chunks:
            if chunk['role'] == role_name:
                role_query = chunk['text']
                break
    
    if not role_query:
        print(f"  ✗ No JD found for role: {role_name}")
        return []
    
    print(f"\nQuery preview: {role_query[:200]}...")
    
    # Search BSC index
    results = semantic_search(bsc_index, bsc_chunks, role_query, top_k=top_k)
    
    print(f"\nTop {top_k} matched BSC objectives:")
    for i, res in enumerate(results):
        print(f"\n  {i+1}. Similarity: {res['similarity']:.3f}")
        print(f"     Strategic Area: {res['chunk'].get('strategic_area', 'N/A')}")
        print(f"     KPI: {res['chunk'].get('kpi', 'N/A')}")
        print(f"     Target: {res['chunk'].get('target', 'N/A')[:100]}")
    
    return results


# ============================================
# STEP 5: BUILD GENERATION PROMPT
# ============================================

def build_generation_prompt(role_name, jd_chunks, matched_bsc_results):
    """
    Build prompt for LLM to generate SMART objectives
    """
    # Get full JD context for the role
    jd_context = ""
    for chunk in jd_chunks:
        if chunk['role'] == role_name:
            jd_context += chunk['text'] + "\n\n"
    
    # Limit JD context length
    if len(jd_context) > 3000:
        jd_context = jd_context[:3000] + "..."
    
    # Format BSC matches
    bsc_text = ""
    for i, res in enumerate(matched_bsc_results[:3]):  # Use top 3 BSC matches
        chunk = res['chunk']
        bsc_text += f"""
BSC Objective {i+1} (Relevance: {res['similarity']:.2f}):
- Strategic Area: {chunk.get('strategic_area', 'N/A')}
- KPI: {chunk.get('kpi', 'N/A')}
- Target: {chunk.get('target', 'N/A')}
"""
    
    prompt = f"""You are an expert in banking performance management creating SMART objectives.

ROLE INFORMATION:
{jd_context}

STRATEGIC GOALS (from BSC that align with this role):
{bsc_text}

Based on the role responsibilities and the strategic goals above, generate 3-5 SMART objectives for this role.

Each objective must follow the SMART criteria:
- Specific: Clear and detailed
- Measurable: Has a metric or target
- Achievable: Realistic for this role
- Relevant: Directly linked to the strategic goal
- Time-bound: Has a deadline (e.g., "by Q4 2025")

Format each objective as:
**Objective 1:** [SMART objective text]
**Objective 2:** [SMART objective text]

Make sure each objective is practical for a banking branch role.
"""
    return prompt


# ============================================
# STEP 6: GENERATE SMART OBJECTIVES
# ============================================

def generate_smart_objectives(role_name, jd_chunks, bsc_index, bsc_chunks, llm_function):
    """
    Complete pipeline: Match BSC → Generate SMART objectives
    """
    print(f"\n{'='*60}")
    print(f"GENERATING SMART OBJECTIVES FOR: {role_name}")
    print(f"{'='*60}")
    
    # Step 1: Match role to BSC
    print("\n[1/2] Matching role to BSC objectives...")
    matches = match_role_to_bsc(role_name, jd_chunks, bsc_index, bsc_chunks, top_k=3)
    
    if not matches:
        print("  ✗ No BSC matches found")
        return None
    
    # Step 2: Build prompt
    print("\n[2/2] Generating SMART objectives...")
    prompt = build_generation_prompt(role_name, jd_chunks, matches)
    
    # Step 3: Call LLM
    response = llm_function(prompt)
    
    return {
        'role': role_name,
        'matched_bsc': [(m['chunk'], m['similarity']) for m in matches],
        'generated_objectives': response
    }


# ============================================
# STEP 7: BUILD ALL INDEXES AND RUN
# ============================================

def build_bsc_jd_indexes(bsc_chunks, jd_chunks, output_dir="./indexes"):
    """
    Build FAISS indexes for BSC and JD only
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("=" * 60)
    print("BUILDING FAISS INDEXES (BSC + JD)")
    print("=" * 60)
    
    indexes = {}
    
    # 1. BSC Index
    if bsc_chunks:
        bsc_index, _ = create_faiss_index(bsc_chunks, "BSC")
        save_index(bsc_index, bsc_chunks, f"{output_dir}/bsc_index")
        indexes['bsc'] = (bsc_index, bsc_chunks)
    
    # 2. JD Index
    if jd_chunks:
        jd_index, _ = create_faiss_index(jd_chunks, "JD")
        save_index(jd_index, jd_chunks, f"{output_dir}/jd_index")
        indexes['jd'] = (jd_index, jd_chunks)
    
    # 3. Combined Index (optional)
    all_chunks = []
    if bsc_chunks:
        all_chunks.extend(bsc_chunks)
    if jd_chunks:
        all_chunks.extend(jd_chunks)
    
    if all_chunks:
        combined_index, _ = create_faiss_index(all_chunks, "Combined")
        save_index(combined_index, all_chunks, f"{output_dir}/combined_index")
        indexes['combined'] = (combined_index, all_chunks)
    
    print("\n" + "=" * 60)
    print("INDEX BUILDING COMPLETE!")
    print("=" * 60)
    
    return indexes


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    
    # Assuming you already have these from your chunking code:
    # bsc_chunks = list of BSC chunks from chunk_bsc_enhanced()
    # jd_chunks = list of JD chunks from chunk_all_job_descriptions_simple()
    
    # For testing with sample data (replace with your actual chunks)
    # bsc_chunks = [...]  
    # jd_chunks = [...]
    
    # Step 1: Build indexes
    # indexes = build_bsc_jd_indexes(bsc_chunks, jd_chunks)
    
    # Step 2: Test matching for a specific role
    # test_role = "branch manager special branch"
    # matches = match_role_to_bsc(test_role, jd_chunks, indexes['bsc'][0], indexes['bsc'][1])
    
    # Step 3: Generate SMART objectives (requires LLM)
    # def my_llm(prompt):
    #     # Your LLM call here (OpenAI, Ollama, etc.)
    #     return "Generated SMART objectives..."
    
    # result = generate_smart_objectives(test_role, jd_chunks, indexes['bsc'][0], indexes['bsc'][1], my_llm)
    
    print("""
    ╔══════════════════════════════════════════════════════════════╗
    ║                    NEXT STEPS (BSC + JD ONLY)                ║
    ╠══════════════════════════════════════════════════════════════╣
    ║                                                              ║
    ║  ✅ Step 1: Chunk BSC (COMPLETE)                             ║
    ║  ✅ Step 2: Chunk JD (COMPLETE)                              ║
    ║                                                              ║
    ║  ➡️ Step 3: Run build_bsc_jd_indexes(bsc_chunks, jd_chunks)  ║
    ║                                                              ║
    ║  ➡️ Step 4: Test matching with match_role_to_bsc()           ║
    ║                                                              ║
    ║  ➡️ Step 5: Connect your LLM (Ollama, OpenAI, etc.)         ║
    ║                                                              ║
    ║  ➡️ Step 6: Run generate_smart_objectives()                  ║
    ║                                                              ║
    ║  ➡️ Step 7: Review and refine generated objectives          ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    """)

In [ ]:
all_chunks = jd_context + bsc_chunks

['Role: branch manager special branch\n        Job Objective: to ensure the overall banking operation activities of the branch performed as per the bank’s policy, procedure, guideline and other pertinent regulation to achieve the bank’s vision and strategic objectives.\n        job category: middle level management: mlm\n        job grade: 15 unit: branch\n        job department: district\n        job division: retail & branch banking',
 'Role: branch manager special branch\n                Responsibility: the job holder’s responsibilities include, but are not limited to: plan, coordinate and manage the overall banking operation activities in the branch; set annual operational plan and operational budget of the branch for retail, wholesale and ifb businesses; and digital banking channels; create awareness and clarity to staff of the branch about the bank’s strategy, plans and corporate views; ensure required resources are satisfied at the branch; conduct daily sales and operational hud

{'job_title': 'branch manager special branch',
 'division': 'retail & branch banking',
 'department': 'district',
 'job_grade': '15 unit: branch',
 'job_category': 'middle level management: mlm',
 'job_objective': 'to ensure the overall banking operation activities of the branch performed as per the bank’s policy, procedure, guideline and other pertinent regulation to achieve the bank’s vision and strategic objectives.',
 'Key_Job_Duties_&_Responsibilities': "the job holder’s responsibilities include, but are not limited to: interact directly with customers throughout all phases of the sales process; identify customers’ need, propose/plan relevant products or services and ensure that customers have positive experiences from start to finish; manage relationships with customers, serving as a key point of contact, from initial lead outreach to when purchase is ultimately made; make preparation, approach customers, make presentation, handle objections and close sales as the selling process


Job Title: branch manager special branch
Division: retail & branch banking
Department: district
Job Grade: 15 unit: branch
Job Category: middle level management: mlm

Job Objective:
to ensure the overall banking operation activities of the branch performed as per the bank’s policy, procedure, guideline and other pertinent regulation to achieve the bank’s vision and strategic objectives.

Key Job Duties & Responsibilities:
the job holder’s responsibilities include, but are not limited to: interact directly with customers throughout all phases of the sales process; identify customers’ need, propose/plan relevant products or services and ensure that customers have positive experiences from start to finish; manage relationships with customers, serving as a key point of contact, from initial lead outreach to when purchase is ultimately made; make preparation, approach customers, make presentation, handle objections and close sales as the selling process passes through steps (starting from 